## AVAIL trasnaction data

In [ ]:
import requests
import json
import time

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


In [ ]:
output_dir = DATA_DIR / "raw" / "avail" / "blocks"
output_dir.mkdir(parents=True, exist_ok=True)

url = "https://avail.api.subscan.io/api/scan/block"
num = 246001
block_batch = []
batch_size = 1000  # 한번에 저장할 블록 수
headers = {
   'User-Agent': 'Apidog/1.0.0 (https://apidog.com)',
   'Content-Type': 'application/json'
}

while True:
    payload = json.dumps({
        "block_num": num,
        "only_head": False
    })

    response = requests.post(url, headers=headers, data=payload)

    if response.status_code == 200:
        response_data = response.json()
        block_batch.append(response_data)

        # 1000개씩 저장
        if len(block_batch) == batch_size:
            start_num = num - batch_size + 1
            end_num = num
            file_name = output_dir / f"AVAIL_blocks_{start_num}_to_{end_num}.json"
            
            with file_name.open('w') as json_file:
                json.dump(block_batch, json_file, indent=4)
            
            print(f"Blocks {start_num} to {end_num} saved to {file_name}")

            # 저장 후 리스트 초기화
            block_batch = []

        # 다음 블록 번호로 이동
        num += 1

    else:
        print(f"Error occurred: {response.status_code} - {response.text}")
        
        # 1000개 미만의 블록이 남아있을 경우 마지막으로 저장
        if block_batch:
            start_num = num - len(block_batch)
            end_num = num - 1
            file_name = output_dir / f"AVAIL_blocks_{start_num}_to_{end_num}.json"
            
            with file_name.open('w') as json_file:
                json.dump(block_batch, json_file, indent=4)
            
            print(f"Blocks {start_num} to {end_num} saved to {file_name}")

        break

    time.sleep(0.8)


대략 만개에 2시간정도 걸린다. 만개에 2틀정도. 총 블록의 수가 24만개이므로 그렇게 오래걸릴 것 같지는 않다.